# 03 — Construire les fenêtres glissantes

**Objectif :** transformer une suite d'identifiants en exemples de prédiction du prochain token.

Nous commençons avec des entiers choisis à la main, puis nous utilisons le tokenizer et les partitions préparés dans les notebooks précédents. Nous manipulons seulement des listes Python : aucun modèle n'est entraîné et PyTorch n'est pas nécessaire.

À la fin, tu sauras expliquer :

- pourquoi une entrée de longueur **L** demande **L + 1** identifiants disponibles ;
- comment le décalage de la cible et le pas `stride` fonctionnent ;
- comment grouper plusieurs exemples en un batch ;
- comment préserver les frontières entre documents et entre entraînement et validation.

Les exemples numériques sont inventés pour apprendre. Les fichiers réels seront chargés dans la seconde partie ; ce notebook ne télécharge rien.

## 1. Une suite d'identifiants

Les nombres ci-dessous représentent des tokens. Leur valeur numérique ne donne aucune information sur leur sens.

Pour chaque position, le modèle devra prédire le nombre suivant. Ici, nous préparons uniquement les entrées et les cibles.

In [ ]:
ids = [10, 20, 30, 40, 50, 60]
length = 3

x = ids[:length]
y = ids[1:length + 1]

print("Entrée :", x)  # [10, 20, 30]
print("Cible  :", y)  # [20, 30, 40]
assert len(x) == len(y) == length
assert x[1:] == y[:-1]

### Pourquoi L + 1 ?

Pour une entrée `[10, 20, 30]`, les trois réponses attendues sont `[20, 30, 40]`. Le quatrième identifiant est nécessaire pour la dernière cible.

Le **pas** `stride` indique de combien de positions avancer avant la fenêtre suivante. Il ne change jamais le décalage de la cible, qui reste égal à **un token**.

Dans `range(0, len(ids) - length, stride)`, la borne de droite est exclue. Cela garantit que la dernière cible existe.

In [ ]:
def windows(ids, length=8, stride=4):
    """Produire des paires (entrée, cible) complètes à partir d'une séquence."""
    if length < 1 or stride < 1:
        raise ValueError("length et stride doivent être positifs.")

    for start in range(0, len(ids) - length, stride):
        x = ids[start:start + length]
        y = ids[start + 1:start + length + 1]
        yield x, y

In [ ]:
examples = list(windows(ids, length=3, stride=2))
assert examples == [
    ([10, 20, 30], [20, 30, 40]),
    ([30, 40, 50], [40, 50, 60]),
]

for number, (x, y) in enumerate(examples, start=1):
    print(f"Fenêtre {number} : X = {x} ; Y = {y}")

# Avec L identifiants, il manque la dernière cible.
assert list(windows([10, 20, 30], length=3, stride=1)) == []

# Avec L + 1 identifiants, une fenêtre est possible.
assert list(windows([10, 20, 30, 40], length=3, stride=1)) == [
    ([10, 20, 30], [20, 30, 40])
]

## 2. Observer le pas

Avec `stride=1`, les fenêtres se chevauchent fortement. Avec `stride=length`, les blocs d'entrée successifs ne se chevauchent pas ; leurs cibles restent décalées d'un token.

Cette version conserve uniquement les fenêtres complètes. Un morceau trop court à la fin n'est pas complété artificiellement. Un pas plus grand peut laisser des positions inutilisées.

In [ ]:
sequence = list(range(9))

for stride in (1, 2, 3):
    result = list(windows(sequence, length=3, stride=stride))
    print(f"stride={stride} : {len(result)} fenêtres")
    for x, y in result:
        print("  X =", x, "Y =", y)

## 3. Grouper les exemples en batches

Un **batch** est un groupe d'exemples. Pour un batch de taille B et des fenêtres de longueur L, `X` et `Y` sont ici des listes contenant B listes de L entiers.

Le dernier batch peut être plus petit que B. Le paramètre `drop_last=True` permet de l'écarter. Il n'y a ni calcul de prédiction ni mise à jour de paramètres à cette étape.

In [ ]:
def batches(examples, batch_size=4, drop_last=False):
    """Grouper des paires (x, y), sans modifier les fenêtres."""
    if batch_size < 1:
        raise ValueError("batch_size doit être positif.")

    batch_x, batch_y = [], []
    for x, y in examples:
        batch_x.append(x)
        batch_y.append(y)
        if len(batch_x) == batch_size:
            yield batch_x, batch_y
            batch_x, batch_y = [], []

    if batch_x and not drop_last:
        yield batch_x, batch_y

In [ ]:
numeric_examples = list(windows(list(range(8)), length=3, stride=1))
numeric_batches = list(batches(numeric_examples, batch_size=2))

assert [len(batch_x) for batch_x, _ in numeric_batches] == [2, 2, 1]

first_x, first_y = numeric_batches[0]
print("X :", first_x)
print("Y :", first_y)

assert first_x == [[0, 1, 2], [1, 2, 3]]
assert first_y == [[1, 2, 3], [2, 3, 4]]

## 4. Exercices avant la correction

1. Pour `[5, 6, 7, 8, 9, 10, 11]`, écris les fenêtres avec `length=3` et `stride=2`.
2. Une **séquence finale de trois identifiants, EOS déjà compris**, suffit-elle à produire une fenêtre de longueur 3 ?
3. Pourquoi changer `stride` ne change-t-il pas le décalage entre entrée et cible ?
4. Tu disposes de cinq exemples et d'un batch de taille 2. Combien de batches obtiens-tu avec et sans `drop_last=True` ?

Essaie de répondre avant d'exécuter la cellule suivante.

In [ ]:
answer = list(windows([5, 6, 7, 8, 9, 10, 11], length=3, stride=2))
assert answer == [
    ([5, 6, 7], [6, 7, 8]),
    ([7, 8, 9], [8, 9, 10]),
]
print("1. Fenêtres :", answer)
print("2. Non : il faut quatre identifiants disponibles pour L=3.")
print("3. stride choisit les débuts des fenêtres ; la cible reste décalée de 1.")

five_examples = list(windows(list(range(8)), length=3, stride=1))
kept = list(batches(five_examples, batch_size=2))
dropped = list(batches(five_examples, batch_size=2, drop_last=True))
assert len(kept) == 3 and len(dropped) == 2
print("4. Trois batches (2, 2, 1), ou deux batches (2, 2).")

## 5. Retrouver les fichiers du projet

Nous utilisons maintenant les fichiers préparés dans les notebooks précédents :

- `artifacts/tokenizer/tokenizer_darija.json` ;
- `data/processed/train.jsonl` ;
- `data/processed/validation.jsonl`.

Chaque ligne JSONL doit être un objet contenant un champ `text` de type chaîne. Les partitions sont déjà définies : **nous ne mélangeons pas les documents et nous ne refaisons pas la séparation ici**.

Le noyau doit déjà disposer de la bibliothèque `tokenizers` utilisée dans le notebook 02. Aucun téléchargement ni apprentissage du tokenizer n'est effectué dans ce notebook.

In [ ]:
import json
import unicodedata
from pathlib import Path


def find_project_root(start=None):
    """Chercher README.md et notebooks/ depuis cwd, puis dans ses parents."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").is_file()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Racine introuvable : ouvre ce notebook depuis le dépôt "
        "contenant README.md et notebooks/."
    )


PROJECT_ROOT = find_project_root()
TOKENIZER_PATH = PROJECT_ROOT / "artifacts/tokenizer/tokenizer_darija.json"
TRAIN_PATH = PROJECT_ROOT / "data/processed/train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data/processed/validation.jsonl"

missing = [
    str(path.relative_to(PROJECT_ROOT))
    for path in (TOKENIZER_PATH, TRAIN_PATH, VALIDATION_PATH)
    if not path.is_file()
]
if missing:
    raise FileNotFoundError(
        "Fichiers manquants : " + ", ".join(missing)
        + ". Prépare-les avec les notebooks précédents avant de poursuivre."
    )

print("Racine du projet :", PROJECT_ROOT)

### Charger les règles existantes

`encode_special_tokens=True` est rétabli après le chargement : les chaînes littérales comme `[EOS]` restent du texte ordinaire. Avec Tokenizers 0.22.1, ce drapeau n'est pas conservé dans le fichier JSON.

Le vrai marqueur EOS sera ajouté **par son identifiant**, après l'encodage de chaque document. Réserver un token spécial dans le vocabulaire ne l'ajoute pas automatiquement au texte.

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.encode_special_tokens = True

eos_id = tokenizer.token_to_id("[EOS]")
if eos_id is None:
    raise ValueError("Le tokenizer doit réserver un token spécial [EOS].")

print("Taille du vocabulaire :", tokenizer.get_vocab_size())
print("Identifiant EOS :", eos_id)

## 6. Lire les partitions sans les confondre

Nous lisons les fichiers locaux et conservons leurs frontières de documents. NFC est appliqué de manière identique aux deux partitions ; aucun vocabulaire n'est appris.

L'intersection ci-dessous détecte les documents exactement identiques après NFC. Elle ne détecte pas les paraphrases ou quasi-doublons : leur regroupement relève de la préparation du corpus.

In [ ]:
def read_documents(path):
    """Lire les textes d'un JSONL local, une ligne par document."""
    with Path(path).open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict) or not isinstance(
                record.get("text"), str
            ):
                raise ValueError(
                    f"{Path(path).name}, ligne {line_number} : "
                    "un objet avec un champ text de type chaîne est attendu."
                )
            text = unicodedata.normalize("NFC", record["text"])
            if not text.strip():
                raise ValueError(
                    f"{Path(path).name}, ligne {line_number} : texte vide."
                )
            yield text


train_documents = list(read_documents(TRAIN_PATH))
validation_documents = list(read_documents(VALIDATION_PATH))

if not train_documents or not validation_documents:
    raise ValueError("Les deux partitions doivent contenir des documents.")

overlap = set(train_documents).intersection(validation_documents)
if overlap:
    raise ValueError(
        f"{len(overlap)} document(s) identique(s) dans les deux partitions. "
        "Corrige la préparation du corpus avant de continuer."
    )

print("Documents d'entraînement :", len(train_documents))
print("Documents de validation :", len(validation_documents))

## 7. Construire les fenêtres document par document

Pour chaque document :

1. encoder le texte avec les règles existantes ;
2. ajouter un seul véritable identifiant EOS à la fin ;
3. produire ses fenêtres complètes ;
4. passer au document suivant.

Nous ne concaténons jamais les identifiants de deux documents. Une cible du premier document ne peut donc pas être le début du suivant.

Un texte de trois tokens peut produire une fenêtre de longueur 3 **après ajout d'EOS**, car la séquence comporte alors quatre identifiants. C'est différent de l'exercice où EOS était déjà compris.

In [ ]:
def document_windows(tokenizer, documents, length=8, stride=4):
    """Encoder et découper chaque document séparément, avec un EOS final."""
    eos = tokenizer.token_to_id("[EOS]")
    if eos is None:
        raise ValueError("Le tokenizer ne contient pas [EOS].")

    for text in documents:
        ids = tokenizer.encode(text, add_special_tokens=False).ids
        ids = ids + [eos]
        yield from windows(ids, length=length, stride=stride)

## 8. Préparer un batch de chaque partition

Les mêmes longueur et pas sont appliqués aux deux partitions. Les fenêtres d'entraînement et de validation restent dans des générateurs séparés.

Nous prenons seulement le premier batch de chaque générateur pour observer sa structure. Les cellules suivantes afficheront des résultats issus de tes fichiers quand tu les exécuteras ; aucune valeur réelle n'est préremplie.

In [ ]:
LENGTH = 16
STRIDE = 8
BATCH_SIZE = 4

train_examples = document_windows(
    tokenizer, train_documents, length=LENGTH, stride=STRIDE
)
validation_examples = document_windows(
    tokenizer, validation_documents, length=LENGTH, stride=STRIDE
)

first_train_batch = next(batches(train_examples, BATCH_SIZE), None)
first_validation_batch = next(batches(validation_examples, BATCH_SIZE), None)

for name, batch in (
    ("Entraînement", first_train_batch),
    ("Validation", first_validation_batch),
):
    if batch is None:
        print(name, ": aucune fenêtre complète avec cette longueur.")
        continue

    batch_x, batch_y = batch
    assert len(batch_x) == len(batch_y)
    for x, y in zip(batch_x, batch_y):
        assert len(x) == len(y) == LENGTH
        assert x[1:] == y[:-1]
        assert eos_id not in x
        assert eos_id not in y[:-1]

    print(name, ": forme des listes =", (len(batch_x), LENGTH))
    print("Première entrée :", batch_x[0])
    print("Première cible  :", batch_y[0])

## Ce qu'il faut retenir

- La cible est toujours décalée d'un token ; `stride` règle les débuts des fenêtres.
- Les partitions restent séparées, et aucune fenêtre ne traverse une frontière de document.
- Les textes trop courts ne produisent aucune fenêtre complète.
- Le pas peut laisser une fin de document inutilisée. Ajouter EOS ne garantit donc pas qu'il apparaisse dans les cibles conservées. Avec `stride=1`, la dernière fenêtre complète atteint EOS si le document est assez long.
- Une fenêtre tronquée peut couper les octets d'un caractère : nous vérifions ici les listes d'identifiants, sans exiger que chaque fenêtre décodée isolément restitue un texte complet.
- Ces batches sont des données préparées pour un futur modèle. Ils ne constituent ni une prédiction ni un entraînement.

**Pour vérifier ta compréhension :** si `LENGTH` augmente, pourquoi certains documents cessent-ils de fournir une fenêtre ? Parce que chaque exemple doit disposer de `LENGTH + 1` identifiants, EOS compris.

La prochaine étape sera d'associer des vecteurs aux identifiants avec les embeddings.